# 19 - D15: the leaf check ("is this a leaf at all?")

**Why:** the disease model only knows 11 disease classes, so a photo of anything else (a
shoe, a wall) still gets the "least wrong" disease, and could reach an extension officer.
The supervisor saw exactly this. The leaf check runs **before** the disease model and stops
photos that are not leaves.

**Design (decided 23 Oct 2026, before any photo was scored; `src/leaf_check.py`):** a
separate model that never changes: an ImageNet network, **frozen**, turns each photo into a
summary, and **one logistic-regression layer** answers "leaf or not". Retraining the disease
model can never change it. MobileNetV3 is used because it is the cheapest (about 30 ms per
photo on CPU); the leaf/not-leaf task is easy for any ImageNet network.

**Data:**

| Use | Leaf photos | Non-leaf photos |
|---|---|---|
| train the layer | 5,000 sampled from `train_d22.csv` (every source and class) | COCO 2017 validation, 60% of the everyday photos |
| set the cut-off | the validation split | 20% of COCO |
| report only | test split, PlantDoc, the selection set | 20% of COCO; COCO plant-like, fruit and food photos; the author's own photos and screenshots |

COCO (Lin et al., 2014): annotations CC BY 4.0; each photo is Flickr's, under its own
licence, recorded per photo. Photos are only used to fit the layer; only the layer (about
1,000 numbers) ships with the app.

**Rules (fixed before running):** the cut-off is the highest leaf probability at which at
least **99% of validation leaf photos pass** (blocking a real farmer costs more than letting
an odd photo through, which the disease model's low confidence already refers). The check is
used only if it also rejects at least **95% of validation non-leaf photos**. If MobileNetV3
does not, EfficientNet-B0 is tried the same way; if neither does, the check is reported and
stays off.

**How to run:** CPU is enough (nothing deep is trained). *Run all*. About 30-40 minutes, most
of it copying the leaf photos and downloading COCO (1 GB). Optional: the author's own photos
in `CropDiseaseDiagnosis/datasets/raw/leafcheck_hard/leaf/` and `.../not_leaf/`.

In [ ]:
from pathlib import Path
import sys

from google.colab import drive

drive.mount("/content/drive")

REPO_URL = "https://github.com/BilliartMuthoni/confidence-calibrated-crop-diagnosis.git"
REPO_DIR = Path("/content/drive/MyDrive/Sproutly-app")

if not REPO_DIR.exists():
    !git clone {REPO_URL} {REPO_DIR}

ML_DIR = REPO_DIR / "ml"
if str(ML_DIR) not in sys.path:
    sys.path.insert(0, str(ML_DIR))

import config

In [ ]:
!git -C {REPO_DIR} config core.fileMode false
!git -C {REPO_DIR} pull --ff-only
!git -C {REPO_DIR} status --short --branch

In [ ]:
%pip install -q -r {ML_DIR}/requirements.txt

## Settings

In [ ]:
import importlib
import json
import shutil
import urllib.request
import zipfile

import numpy as np
import pandas as pd
import torch

from src import leaf_check, training
from src.data import image_dataset
for module in (leaf_check, training, image_dataset):
    importlib.reload(module)

SEED = config.RANDOM_SEED
LEAF_SAMPLE = 5000
COCO_IMAGES = "http://images.cocodataset.org/zips/val2017.zip"
COCO_ANNOTATIONS = "http://images.cocodataset.org/annotations/annotations_trainval2017.zip"

OUT_DIR = config.MODELS_DIR / "leaf_check"
REPO_RESULTS = ML_DIR / "results" / "leaf_check"
HARD_DIR = config.RAW_DATA_DIR / "leafcheck_hard"    # optional: the author's own photos
LOCAL = Path("/content/leafcheck")                   # copies and COCO, Colab disk only
OUT_DIR.mkdir(parents=True, exist_ok=True)
REPO_RESULTS.mkdir(parents=True, exist_ok=True)
LOCAL.mkdir(parents=True, exist_ok=True)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
torch.set_num_threads(max(torch.get_num_threads(), 2))
print("Running on", DEVICE, "| pass rate", leaf_check.PASS_RATE, "| reject target", leaf_check.REJECT_TARGET)

## Leaf photos

Only paths are listed here; the photos are copied (shrunk, as for training) to the Colab disk.

In [ ]:
splits = {name: pd.read_csv(config.SPLITS_DIR / f"{name}.csv") for name in ["train_d22", "val", "test", "ood_test", "selection"]}
train = splits["train_d22"]
# The same share of every source and class, so field photos are as present as in training.
leaf_train = train.groupby(["source", "class"], group_keys=False).apply(
    lambda g: g.sample(frac=min(1.0, LEAF_SAMPLE / len(train)), random_state=SEED))
LEAF = {"train": leaf_train, "val": splits["val"], "test": splits["test"], "plantdoc": splits["ood_test"],
        "selection": splits["selection"]}
display(pd.Series({name: len(rows) for name, rows in LEAF.items()}, name="leaf photos"))

all_paths = sorted(set(pd.concat(LEAF.values())["path"]))
cached = image_dataset.cache_images(all_paths, config.RAW_DATA_DIR, LOCAL / "leaves")
assert cached == len(all_paths), f"only {cached} of {len(all_paths)} leaf photos found"

## Non-leaf photos: COCO 2017 validation

In [ ]:
def fetch(url, dest):
    if not dest.exists():
        urllib.request.urlretrieve(url, dest)
    return dest


for url in [COCO_IMAGES, COCO_ANNOTATIONS]:
    archive = fetch(url, LOCAL / Path(url).name)
    target = LOCAL / Path(url).stem
    if not target.exists():
        with zipfile.ZipFile(archive) as z:
            members = [m for m in z.namelist() if "annotations" not in url or m.endswith("instances_val2017.json")]
            z.extractall(target, members)

coco = leaf_check.coco_images(LOCAL / "annotations_trainval2017" / "annotations" / "instances_val2017.json")
coco["path"] = [LOCAL / "val2017" / "val2017" / f for f in coco["file"]]
assert all(p.exists() for p in coco["path"]), "some COCO photos are missing"
coco["split"] = leaf_check.split_everyday(coco, SEED)
display(pd.crosstab(coco["group"], coco["split"]))
display(coco["licence"].value_counts().rename("COCO photos per Flickr licence"))

In [ ]:
# The author's own photos and screenshots (optional; report only).
hard = []
for label in ["leaf", "not_leaf"]:
    folder = HARD_DIR / label
    if folder.exists():
        hard += [{"path": p, "really leaf": label == "leaf"} for p in sorted(folder.rglob("*"))
                 if p.suffix.lower() in config.SUPPORTED_EXTENSIONS]
hard = pd.DataFrame(hard, columns=["path", "really leaf"])
print(len(hard), "of the author's own photos:", hard["really leaf"].value_counts().to_dict() if len(hard) else "none")

## Summaries from the frozen network, then the leaf layer

Summaries are saved on Drive, so a rerun skips this step.

In [ ]:
def summaries(architecture):
    saved = OUT_DIR / f"{architecture}_features.npz"
    if saved.exists():
        return dict(np.load(saved, allow_pickle=True))
    model = leaf_check.backbone(architecture)
    out = {}
    for name, rows in LEAF.items():
        out[f"leaf_{name}"], ok = leaf_check.features(model, [LOCAL / "leaves" / p for p in rows["path"]], DEVICE)
        assert ok.all(), f"{(~ok).sum()} unreadable leaf photos in {name}"
    out["coco"], ok = leaf_check.features(model, list(coco["path"]), DEVICE)
    assert ok.all(), f"{(~ok).sum()} unreadable COCO photos"
    if len(hard):
        out["hard"], out["hard_ok"] = leaf_check.features(model, list(hard["path"]), DEVICE)
    np.savez(saved, **out)
    return out


def try_architecture(architecture):
    f = summaries(architecture)
    split = coco["split"].to_numpy()
    head = leaf_check.fit(f["leaf_train"], f["coco"][split == "train"], SEED)
    threshold = leaf_check.threshold_for_pass_rate(leaf_check.leaf_probability(head, f["leaf_val"]))
    rejected_val = float((leaf_check.leaf_probability(head, f["coco"][split == "val"]) < threshold).mean())
    return {"architecture": architecture, "head": head, "threshold": threshold, "features": f,
            "val leaves passed": float((leaf_check.leaf_probability(head, f["leaf_val"]) >= threshold).mean()),
            "val non-leaves rejected": rejected_val, "meets rule": rejected_val >= leaf_check.REJECT_TARGET}


TRIED = []
for architecture in leaf_check.ARCHITECTURES:
    result = try_architecture(architecture)
    TRIED.append(result)
    print(f"{architecture}: cut-off {result['threshold']:.4f} | validation leaves passed {result['val leaves passed']:.2%} "
          f"| validation non-leaves rejected {result['val non-leaves rejected']:.2%} | meets the rule: {result['meets rule']}")
    if result["meets rule"]:
        break
CHOSEN = next((r for r in TRIED if r["meets rule"]), None)
print("\nLEAF CHECK:", f"ON, {CHOSEN['architecture']}" if CHOSEN else "OFF: no network met the rule (reported)")

## Every set, reported (nothing here was used to choose)

In [ ]:
shown = CHOSEN or TRIED[-1]
f, head, threshold = shown["features"], shown["head"], shown["threshold"]
split = coco["split"].to_numpy()
group = coco["group"].to_numpy()
sets = {
    "leaves: test split": (f["leaf_test"], True),
    "leaves: PlantDoc (internet field photos)": (f["leaf_plantdoc"], True),
    "leaves: selection set (unseen field sources)": (f["leaf_selection"], True),
    "COCO everyday, held-out test": (f["coco"][split == "test"], False),
    "COCO plant-like (potted plant, broccoli)": (f["coco"][group == "plant-like"], False),
    "COCO fruit (apple, orange, banana)": (f["coco"][group == "fruit"], False),
    "COCO food": (f["coco"][group == "food"], False),
}
if len(hard):
    ok = f["hard_ok"].astype(bool)
    is_leaf = hard["really leaf"].to_numpy()
    sets["own photos: leaves"] = (f["hard"][ok & is_leaf], True)
    sets["own photos: not leaves"] = (f["hard"][ok & ~is_leaf], False)
report = leaf_check.evaluate(head, threshold, sets)
display(report.round(4))

by_source = pd.DataFrame([{"source": s, "photos": int(m.sum()), "passed as leaf": float(
    (leaf_check.leaf_probability(head, f["leaf_selection"][m]) >= threshold).mean())}
    for s in sorted(LEAF["selection"]["source"].unique()) for m in [(LEAF["selection"]["source"] == s).to_numpy()]])
display(by_source.set_index("source").round(4))

## Look at the mistakes

Leaf photos the check would block, and non-leaf photos it would let through: these show what
it has not learned, and are worth a look before trusting it.

In [ ]:
import matplotlib.pyplot as plt
from PIL import Image


def grid(paths, scores, title, file, n=12):
    if not len(paths):
        print(title, ": none")
        return
    order = np.argsort(scores)[:n] if "blocked" in title else np.argsort(-scores)[:n]
    fig, axes = plt.subplots(2, 6, figsize=(14, 5))
    for ax in axes.ravel():
        ax.axis("off")
    for ax, i in zip(axes.ravel(), order):
        with Image.open(paths[i]) as image:
            ax.imshow(image.convert("RGB"))
        ax.set_title(f"leaf p = {scores[i]:.2f}", fontsize=8)
    fig.suptitle(title, x=0.01, ha="left", fontsize=11)
    plt.tight_layout()
    plt.savefig(OUT_DIR / file, dpi=110)
    plt.show()


for name in ["selection", "plantdoc", "test"]:
    p = leaf_check.leaf_probability(head, f[f"leaf_{name}"])
    blocked = p < threshold
    paths = [LOCAL / "leaves" / x for x in LEAF[name]["path"][blocked]]
    grid(paths, p[blocked], f"Leaf photos blocked ({name})", f"blocked_leaves_{name}.png")

p = leaf_check.leaf_probability(head, f["coco"])
through = (p >= threshold) & (split != "train")
grid(list(coco["path"][through]), p[through], "Non-leaf photos let through (COCO, not used for training)", "passed_non_leaves.png")
if len(hard):
    ok = f["hard_ok"].astype(bool)
    p = np.full(len(hard), np.nan)
    p[ok] = leaf_check.leaf_probability(head, f["hard"][ok])
    own = hard.assign(**{"leaf probability": p, "passed": p >= threshold})
    own["right"] = own["passed"] == own["really leaf"]
    display(own.assign(file=own["path"].map(lambda x: Path(x).name)).drop(columns="path").round(3))

## Save the leaf check for the backend

In [ ]:
summary = {"chosen": CHOSEN["architecture"] if CHOSEN else None, "enabled": CHOSEN is not None,
           "tried": [{k: v for k, v in r.items() if k not in ("head", "features")} for r in TRIED],
           "report": report.reset_index().to_dict(orient="records"),
           "coco_licences": coco["licence"].value_counts().to_dict(),
           "rule": f"cut-off where {leaf_check.PASS_RATE:.0%} of validation leaves pass; used only if it rejects "
                   f">= {leaf_check.REJECT_TARGET:.0%} of validation non-leaves (decided 23 Oct 2026, before scoring)"}
training.save_json(summary, OUT_DIR / "decision.json")
report.to_csv(OUT_DIR / "report.csv")
by_source.to_csv(OUT_DIR / "selection_by_source.csv", index=False)
if CHOSEN:
    leaf_check.export(CHOSEN["head"], CHOSEN["architecture"], CHOSEN["threshold"], OUT_DIR / "leaf_check.json",
                      {"validation_leaves_passed": CHOSEN["val leaves passed"],
                       "validation_non_leaves_rejected": CHOSEN["val non-leaves rejected"]})
    print("Saved leaf_check.json for the backend:", CHOSEN["architecture"], "cut-off", round(CHOSEN["threshold"], 4))

## Copy the results into the repository, sync Drive, save to GitHub

Save the notebook first (Ctrl+S). `leaf_check.json` is small (about 60 KB) and is committed:
the backend reads it from the repository.

In [ ]:
for file in OUT_DIR.glob("*"):
    if file.suffix in {".csv", ".json", ".png"}:
        shutil.copy(file, REPO_RESULTS / file.name)
print(sorted(p.name for p in REPO_RESULTS.iterdir()))
drive.flush_and_unmount()
drive.mount("/content/drive")

In [ ]:
from google.colab import userdata

from src import github_sync

shutil.copy(ML_DIR / "notebooks" / "19_leaf_check.ipynb", REPO_RESULTS / "notebook.ipynb")
print(github_sync.push(
    REPO_DIR,
    branch="colab/leaf-check",
    message="D15: leaf check on a frozen ImageNet network, trained on our leaves against COCO",
    token=userdata.get("GITHUB_TOKEN"),
    name="BilliartMuthoni",
    email=userdata.get("GIT_EMAIL"),
    paths=[str(REPO_RESULTS.relative_to(REPO_DIR))],
))